# Results Analysis - Spatio-Temporal GNN Models

This notebook analyzes the performance metrics of 8 GNN models across 2 experiments and 3 forecast horizons (1, 6, and 12 weeks ahead).

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load data from both experiments
exp1 = pd.read_csv('exp1_final_detailed_metrics.csv')
exp2 = pd.read_csv('exp2_final_detailed_metrics.csv')

# Add experiment identifier
exp1['Experiment'] = 'Exp1'
exp2['Experiment'] = 'Exp2'

# Combine both datasets
combined_df = pd.concat([exp1, exp2], ignore_index=True)

print(f"Exp1 shape: {exp1.shape}")
print(f"Exp2 shape: {exp2.shape}")
print(f"Combined shape: {combined_df.shape}")
print(f"\nColumns: {list(combined_df.columns)}")
print(f"\nModels: {sorted(combined_df['Model'].unique())}")
print(f"Horizons: {sorted(combined_df['Horizon'].unique())}")

Exp1 shape: (24, 9)
Exp2 shape: (24, 9)
Combined shape: (48, 9)

Columns: ['Horizon', 'Model', 'Avg_Node_SMAPE', 'Median_Node_SMAPE', 'Std_Node_SMAPE', 'Avg_Node_RMSE', 'Median_Node_RMSE', 'Std_Node_RMSE', 'Experiment']

Models: ['A3TGCN', 'AGCRN', 'DCRNN', 'GConvLSTM', 'GraphWaveNet', 'MTGNN', 'STConv', 'STGCN']
Horizons: [np.int64(1), np.int64(6), np.int64(12)]


## 1. Overview: Average Performance Across All Metrics

In [3]:
# Display full data with formatting
display_df = combined_df.copy()
display_df = display_df.round(4)
display_df = display_df.sort_values(['Horizon', 'Experiment', 'Avg_Node_SMAPE'])
print("\n=== Full Dataset ===")
display(display_df)


=== Full Dataset ===


,Horizon,Model,Avg_Node_SMAPE,Median_Node_SMAPE,Std_Node_SMAPE,Avg_Node_RMSE,Median_Node_RMSE,Std_Node_RMSE,Experiment
0,1,DCRNN,26.6885,21.9740,16.5209,0.9210,0.7328,0.7743,Exp1
1,1,GraphWaveNet,27.4273,23.1001,16.6757,0.9458,0.7502,0.7933,Exp1
2,1,GConvLSTM,27.5536,23.0868,16.5061,0.9436,0.7374,0.8169,Exp1
3,1,STGCN,28.2553,23.6777,16.9236,0.9596,0.7521,0.8112,Exp1
4,1,AGCRN,28.3899,23.6856,17.2823,0.9548,0.7527,0.7895,Exp1
5,1,STConv,30.2958,26.1595,17.0455,1.0224,0.8105,0.8411,Exp1
6,1,MTGNN,30.4189,26.1794,17.6581,1.0354,0.8278,0.8258,Exp1
7,1,A3TGCN,38.4580,33.4756,20.7101,1.2333,1.0035,0.9703,Exp1
24,1,DCRNN,26.1990,21.3806,16.6022,0.9006,0.7006,0.7753,Exp2
25,1,GraphWaveNet,26.7576,21.9501,16.6219,0.9211,0.7170,0.8012,Exp2


## 2. Best Models Per Horizon (By SMAPE)

In [4]:
# Best models by SMAPE for each horizon and experiment
print("=== TOP 3 MODELS BY AVG_NODE_SMAPE ===\n")

for horizon in sorted(combined_df['Horizon'].unique()):
    print(f"\n{'='*60}")
    print(f"HORIZON {horizon} week(s) ahead")
    print('='*60)
    
    for exp in ['Exp1', 'Exp2']:
        exp_data = combined_df[(combined_df['Horizon'] == horizon) & (combined_df['Experiment'] == exp)]
        exp_data = exp_data.sort_values('Avg_Node_SMAPE')
        
        print(f"\n{exp}:")
        top3 = exp_data.head(3)[['Model', 'Avg_Node_SMAPE', 'Avg_Node_RMSE']]
        for idx, (_, row) in enumerate(top3.iterrows(), 1):
            print(f"  {idx}. {row['Model']:15s} - SMAPE: {row['Avg_Node_SMAPE']:.2f}%  RMSE: {row['Avg_Node_RMSE']:.4f}")
        
        # Worst model
        worst = exp_data.tail(1).iloc[0]
        print(f"  Worst: {worst['Model']:12s} - SMAPE: {worst['Avg_Node_SMAPE']:.2f}%  RMSE: {worst['Avg_Node_RMSE']:.4f}")

=== TOP 3 MODELS BY AVG_NODE_SMAPE ===


HORIZON 1 week(s) ahead

Exp1:
  1. DCRNN           - SMAPE: 26.69%  RMSE: 0.9210
  2. GraphWaveNet    - SMAPE: 27.43%  RMSE: 0.9458
  3. GConvLSTM       - SMAPE: 27.55%  RMSE: 0.9436
  Worst: A3TGCN       - SMAPE: 38.46%  RMSE: 1.2333

Exp2:
  1. DCRNN           - SMAPE: 26.20%  RMSE: 0.9006
  2. GraphWaveNet    - SMAPE: 26.76%  RMSE: 0.9211
  3. GConvLSTM       - SMAPE: 27.85%  RMSE: 0.9545
  Worst: A3TGCN       - SMAPE: 38.82%  RMSE: 1.2521

HORIZON 6 week(s) ahead

Exp1:
  1. DCRNN           - SMAPE: 30.43%  RMSE: 1.0042
  2. GraphWaveNet    - SMAPE: 30.75%  RMSE: 1.0163
  3. AGCRN           - SMAPE: 31.70%  RMSE: 1.0383
  Worst: A3TGCN       - SMAPE: 46.12%  RMSE: 1.4389

Exp2:
  1. GraphWaveNet    - SMAPE: 30.59%  RMSE: 1.0164
  2. DCRNN           - SMAPE: 30.98%  RMSE: 1.0115
  3. AGCRN           - SMAPE: 31.80%  RMSE: 1.0410
  Worst: A3TGCN       - SMAPE: 44.56%  RMSE: 1.3954

HORIZON 12 week(s) ahead

Exp1:
  1. AGCRN           - SMAPE:

## 3. Experiment Comparison: Exp1 vs Exp2

In [5]:
# Compare experiments: which one performs better overall?
comparison = []

for horizon in sorted(combined_df['Horizon'].unique()):
    for model in sorted(combined_df['Model'].unique()):
        exp1_data = combined_df[(combined_df['Horizon'] == horizon) & 
                                 (combined_df['Model'] == model) & 
                                 (combined_df['Experiment'] == 'Exp1')]
        exp2_data = combined_df[(combined_df['Horizon'] == horizon) & 
                                 (combined_df['Model'] == model) & 
                                 (combined_df['Experiment'] == 'Exp2')]
        
        if not exp1_data.empty and not exp2_data.empty:
            smape1 = exp1_data['Avg_Node_SMAPE'].values[0]
            smape2 = exp2_data['Avg_Node_SMAPE'].values[0]
            rmse1 = exp1_data['Avg_Node_RMSE'].values[0]
            rmse2 = exp2_data['Avg_Node_RMSE'].values[0]
            
            comparison.append({
                'Horizon': horizon,
                'Model': model,
                'Exp1_SMAPE': smape1,
                'Exp2_SMAPE': smape2,
                'SMAPE_Diff': smape2 - smape1,  # Positive = Exp1 better
                'Exp1_RMSE': rmse1,
                'Exp2_RMSE': rmse2,
                'RMSE_Diff': rmse2 - rmse1,  # Positive = Exp1 better
                'Better_Exp': 'Exp1' if smape2 > smape1 else 'Exp2'
            })

comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.round(4)

print("=== EXPERIMENT COMPARISON ===")
print(f"\nExp1 wins (lower SMAPE): {(comparison_df['Better_Exp'] == 'Exp1').sum()} times")
print(f"Exp2 wins (lower SMAPE): {(comparison_df['Better_Exp'] == 'Exp2').sum()} times")

print("\n=== Full Comparison Table ===")
display(comparison_df.sort_values(['Horizon', 'SMAPE_Diff']))

=== EXPERIMENT COMPARISON ===

Exp1 wins (lower SMAPE): 12 times
Exp2 wins (lower SMAPE): 12 times

=== Full Comparison Table ===


,Horizon,Model,Exp1_SMAPE,Exp2_SMAPE,SMAPE_Diff,Exp1_RMSE,Exp2_RMSE,RMSE_Diff,Better_Exp
4,1,GraphWaveNet,27.4273,26.7576,-0.6697,0.9458,0.9211,-0.0248,Exp2
6,1,STConv,30.2958,29.6957,-0.6001,1.0224,1.0188,-0.0036,Exp2
2,1,DCRNN,26.6885,26.1990,-0.4895,0.9210,0.9006,-0.0204,Exp2
1,1,AGCRN,28.3899,28.1015,-0.2884,0.9548,0.9548,-0.0000,Exp2
5,1,MTGNN,30.4189,30.3945,-0.0244,1.0354,1.0258,-0.0096,Exp2
7,1,STGCN,28.2553,28.4269,0.1716,0.9596,0.9622,0.0025,Exp1
3,1,GConvLSTM,27.5536,27.8478,0.2943,0.9436,0.9545,0.0109,Exp1
0,1,A3TGCN,38.4580,38.8157,0.3577,1.2333,1.2521,0.0188,Exp1
14,6,STConv,37.1529,33.2498,-3.9032,1.2157,1.0791,-0.1367,Exp2
8,6,A3TGCN,46.1218,44.5558,-1.5660,1.4389,1.3954,-0.0435,Exp2


## 4. Visualizations

In [ ]:
# Plot 1: SMAPE across all models, horizons, and experiments
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, horizon in enumerate(sorted(combined_df['Horizon'].unique())):
    ax = axes[idx]
    
    horizon_data = combined_df[combined_df['Horizon'] == horizon]
    
    # Create grouped bar plot
    x = np.arange(len(horizon_data['Model'].unique()))
    width = 0.35
    
    exp1_data = horizon_data[horizon_data['Experiment'] == 'Exp1'].sort_values('Model')
    exp2_data = horizon_data[horizon_data['Experiment'] == 'Exp2'].sort_values('Model')
    
    models = sorted(horizon_data['Model'].unique())
    exp1_smape = [exp1_data[exp1_data['Model'] == m]['Avg_Node_SMAPE'].values[0] if m in exp1_data['Model'].values else 0 for m in models]
    exp2_smape = [exp2_data[exp2_data['Model'] == m]['Avg_Node_SMAPE'].values[0] if m in exp2_data['Model'].values else 0 for m in models]
    
    bars1 = ax.bar(x - width/2, exp1_smape, width, label='Exp1', alpha=0.8)
    bars2 = ax.bar(x + width/2, exp2_smape, width, label='Exp2', alpha=0.8)
    
    ax.set_xlabel('Model', fontweight='bold')
    ax.set_ylabel('Avg Node SMAPE (%)', fontweight='bold')
    ax.set_title(f'Horizon: {horizon} week(s)', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Highlight best model
    min_idx_exp1 = np.argmin(exp1_smape) if exp1_smape else 0
    min_idx_exp2 = np.argmin(exp2_smape) if exp2_smape else 0
    if exp1_smape:
        bars1[min_idx_exp1].set_edgecolor('darkgreen')
        bars1[min_idx_exp1].set_linewidth(3)
    if exp2_smape:
        bars2[min_idx_exp2].set_edgecolor('darkgreen')
        bars2[min_idx_exp2].set_linewidth(3)

plt.tight_layout()
plt.suptitle('Average Node SMAPE Comparison Across Horizons', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Plot 2: RMSE across all models, horizons, and experiments
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, horizon in enumerate(sorted(combined_df['Horizon'].unique())):
    ax = axes[idx]
    
    horizon_data = combined_df[combined_df['Horizon'] == horizon]
    
    # Create grouped bar plot
    x = np.arange(len(horizon_data['Model'].unique()))
    width = 0.35
    
    exp1_data = horizon_data[horizon_data['Experiment'] == 'Exp1'].sort_values('Model')
    exp2_data = horizon_data[horizon_data['Experiment'] == 'Exp2'].sort_values('Model')
    
    models = sorted(horizon_data['Model'].unique())
    exp1_rmse = [exp1_data[exp1_data['Model'] == m]['Avg_Node_RMSE'].values[0] if m in exp1_data['Model'].values else 0 for m in models]
    exp2_rmse = [exp2_data[exp2_data['Model'] == m]['Avg_Node_RMSE'].values[0] if m in exp2_data['Model'].values else 0 for m in models]
    
    bars1 = ax.bar(x - width/2, exp1_rmse, width, label='Exp1', alpha=0.8, color='coral')
    bars2 = ax.bar(x + width/2, exp2_rmse, width, label='Exp2', alpha=0.8, color='skyblue')
    
    ax.set_xlabel('Model', fontweight='bold')
    ax.set_ylabel('Avg Node RMSE', fontweight='bold')
    ax.set_title(f'Horizon: {horizon} week(s)', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Highlight best model
    min_idx_exp1 = np.argmin(exp1_rmse) if exp1_rmse else 0
    min_idx_exp2 = np.argmin(exp2_rmse) if exp2_rmse else 0
    if exp1_rmse:
        bars1[min_idx_exp1].set_edgecolor('darkgreen')
        bars1[min_idx_exp1].set_linewidth(3)
    if exp2_rmse:
        bars2[min_idx_exp2].set_edgecolor('darkgreen')
        bars2[min_idx_exp2].set_linewidth(3)

plt.tight_layout()
plt.suptitle('Average Node RMSE Comparison Across Horizons', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Plot 3: Heatmaps for SMAPE across Exp1 and Exp2
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, exp in enumerate(['Exp1', 'Exp2']):
    ax = axes[idx]
    
    exp_data = combined_df[combined_df['Experiment'] == exp]
    
    # Pivot to create heatmap matrix (Models x Horizons)
    pivot_data = exp_data.pivot(index='Model', columns='Horizon', values='Avg_Node_SMAPE')
    
    # Create heatmap
    sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax, 
                cbar_kws={'label': 'SMAPE (%)'}, vmin=25, vmax=55)
    
    ax.set_title(f'{exp}: Avg Node SMAPE by Model and Horizon', fontsize=12, fontweight='bold')
    ax.set_xlabel('Horizon (weeks)', fontweight='bold')
    ax.set_ylabel('Model', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Difference heatmap (Exp2 - Exp1) - Positive means Exp1 is better
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Create pivot for both experiments
exp1_pivot = combined_df[combined_df['Experiment'] == 'Exp1'].pivot(index='Model', columns='Horizon', values='Avg_Node_SMAPE')
exp2_pivot = combined_df[combined_df['Experiment'] == 'Exp2'].pivot(index='Model', columns='Horizon', values='Avg_Node_SMAPE')

# Calculate difference (positive = Exp1 better, negative = Exp2 better)
diff_pivot = exp2_pivot - exp1_pivot

# Create heatmap
sns.heatmap(diff_pivot, annot=True, fmt='.2f', cmap='RdBu_r', ax=ax, center=0,
            cbar_kws={'label': 'SMAPE Difference (Exp2 - Exp1)'})

ax.set_title('SMAPE Difference: Exp2 vs Exp1\n(Positive=Exp1 Better, Negative=Exp2 Better)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Horizon (weeks)', fontweight='bold')
ax.set_ylabel('Model', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 5: Line plots showing model performance trends across horizons
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# SMAPE trends
ax = axes[0]
for model in sorted(combined_df['Model'].unique()):
    for exp_style, exp in [('solid', 'Exp1'), ('dashed', 'Exp2')]:
        exp_model_data = combined_df[(combined_df['Model'] == model) & (combined_df['Experiment'] == exp)]
        exp_model_data = exp_model_data.sort_values('Horizon')
        
        label = f'{model}' if exp == 'Exp1' else None
        ax.plot(exp_model_data['Horizon'], exp_model_data['Avg_Node_SMAPE'], 
                marker='o', label=label, linestyle=exp_style, alpha=0.7)

ax.set_xlabel('Horizon (weeks)', fontweight='bold')
ax.set_ylabel('Avg Node SMAPE (%)', fontweight='bold')
ax.set_title('SMAPE Trends Across Horizons\n(Solid=Exp1, Dashed=Exp2)', fontsize=11, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xticks(sorted(combined_df['Horizon'].unique()))

# RMSE trends
ax = axes[1]
for model in sorted(combined_df['Model'].unique()):
    for exp_style, exp in [('solid', 'Exp1'), ('dashed', 'Exp2')]:
        exp_model_data = combined_df[(combined_df['Model'] == model) & (combined_df['Experiment'] == exp)]
        exp_model_data = exp_model_data.sort_values('Horizon')
        
        label = f'{model}' if exp == 'Exp1' else None
        ax.plot(exp_model_data['Horizon'], exp_model_data['Avg_Node_RMSE'], 
                marker='o', label=label, linestyle=exp_style, alpha=0.7)

ax.set_xlabel('Horizon (weeks)', fontweight='bold')
ax.set_ylabel('Avg Node RMSE', fontweight='bold')
ax.set_title('RMSE Trends Across Horizons\n(Solid=Exp1, Dashed=Exp2)', fontsize=11, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xticks(sorted(combined_df['Horizon'].unique()))

plt.tight_layout()
plt.show()

## 5. Statistical Analysis

In [ ]:
# Statistical summary
print("=== STATISTICAL SUMMARY ===\n")

print("Overall Statistics by Experiment:")
print("-" * 60)
for exp in ['Exp1', 'Exp2']:
    exp_data = combined_df[combined_df['Experiment'] == exp]
    print(f"\n{exp}:")
    print(f"  Avg SMAPE: {exp_data['Avg_Node_SMAPE'].mean():.2f}% (±{exp_data['Avg_Node_SMAPE'].std():.2f})")
    print(f"  Min SMAPE: {exp_data['Avg_Node_SMAPE'].min():.2f}% ({exp_data[exp_data['Avg_Node_SMAPE'] == exp_data['Avg_Node_SMAPE'].min()]['Model'].values[0]})")
    print(f"  Max SMAPE: {exp_data['Avg_Node_SMAPE'].max():.2f}% ({exp_data[exp_data['Avg_Node_SMAPE'] == exp_data['Avg_Node_SMAPE'].max()]['Model'].values[0]})")
    print(f"  Avg RMSE:  {exp_data['Avg_Node_RMSE'].mean():.4f} (±{exp_data['Avg_Node_RMSE'].std():.4f})")

print("\n" + "=" * 60)
print("\nStatistics by Horizon:")
print("-" * 60)
for horizon in sorted(combined_df['Horizon'].unique()):
    horizon_data = combined_df[combined_df['Horizon'] == horizon]
    print(f"\nHorizon {horizon} week(s):")
    print(f"  Avg SMAPE: {horizon_data['Avg_Node_SMAPE'].mean():.2f}% (±{horizon_data['Avg_Node_SMAPE'].std():.2f})")
    print(f"  Min SMAPE: {horizon_data['Avg_Node_SMAPE'].min():.2f}%")
    print(f"  Max SMAPE: {horizon_data['Avg_Node_SMAPE'].max():.2f}%")
    print(f"  Avg RMSE:  {horizon_data['Avg_Node_RMSE'].mean():.4f} (±{horizon_data['Avg_Node_RMSE'].std():.4f})")

In [ ]:
# Model performance consistency across horizons
print("=== MODEL CONSISTENCY ANALYSIS ===")
print("(Lower std = more consistent performance across horizons)\n")

model_consistency = []
for model in sorted(combined_df['Model'].unique()):
    for exp in ['Exp1', 'Exp2']:
        model_exp_data = combined_df[(combined_df['Model'] == model) & (combined_df['Experiment'] == exp)]
        
        if len(model_exp_data) > 0:
            smape_std = model_exp_data['Avg_Node_SMAPE'].std()
            smape_mean = model_exp_data['Avg_Node_SMAPE'].mean()
            smape_cv = (smape_std / smape_mean) * 100 if smape_mean > 0 else 0  # Coefficient of variation
            
            model_consistency.append({
                'Model': model,
                'Experiment': exp,
                'Mean_SMAPE': smape_mean,
                'Std_SMAPE': smape_std,
                'CV_SMAPE': smape_cv
            })

consistency_df = pd.DataFrame(model_consistency)
consistency_df = consistency_df.round(2)
consistency_df = consistency_df.sort_values(['Experiment', 'CV_SMAPE'])

print("Consistency Ranking (by Coefficient of Variation):")
display(consistency_df)

In [ ]:
# Median vs Average Analysis
print("=== MEDIAN vs AVERAGE NODE SMAPE ===")
print("(Large differences indicate skewed distributions)\n")

for horizon in sorted(combined_df['Horizon'].unique()):
    print(f"\nHorizon {horizon} week(s):")
    print("-" * 50)
    
    horizon_data = combined_df[combined_df['Horizon'] == horizon].copy()
    horizon_data['Avg_Median_Diff'] = horizon_data['Avg_Node_SMAPE'] - horizon_data['Median_Node_SMAPE']
    horizon_data = horizon_data.sort_values('Avg_Median_Diff', ascending=False)
    
    for _, row in horizon_data.iterrows():
        diff = row['Avg_Median_Diff']
        print(f"  {row['Model']:15s} ({row['Experiment']}): Avg={row['Avg_Node_SMAPE']:.2f}%, "
              f"Median={row['Median_Node_SMAPE']:.2f}%, Diff={diff:.2f}%")

In [ ]:
# Overall model ranking across all horizons
print("=== OVERALL MODEL RANKING ===")
print("(Based on average SMAPE across all horizons)\n")

for exp in ['Exp1', 'Exp2']:
    print(f"\n{exp}:")
    print("-" * 60)
    
    exp_data = combined_df[combined_df['Experiment'] == exp]
    
    # Calculate average performance across all horizons for each model
    model_avg = exp_data.groupby('Model').agg({
        'Avg_Node_SMAPE': 'mean',
        'Avg_Node_RMSE': 'mean',
        'Median_Node_SMAPE': 'mean',
        'Std_Node_SMAPE': 'mean'
    }).reset_index()
    
    model_avg = model_avg.sort_values('Avg_Node_SMAPE')
    
    for rank, (_, row) in enumerate(model_avg.iterrows(), 1):
        print(f"  {rank}. {row['Model']:15s} - SMAPE: {row['Avg_Node_SMAPE']:.2f}%, "
              f"RMSE: {row['Avg_Node_RMSE']:.4f}, StdDev: {row['Std_Node_SMAPE']:.2f}")

## 6. Distribution Analysis

In [ ]:
# Box plots showing distribution of SMAPE across models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By horizon
ax = axes[0]
sns.boxplot(data=combined_df, x='Horizon', y='Avg_Node_SMAPE', hue='Experiment', ax=ax)
ax.set_xlabel('Horizon (weeks)', fontweight='bold')
ax.set_ylabel('Avg Node SMAPE (%)', fontweight='bold')
ax.set_title('SMAPE Distribution by Horizon', fontsize=12, fontweight='bold')
ax.legend(title='Experiment')
ax.grid(axis='y', alpha=0.3)

# By model
ax = axes[1]
sns.boxplot(data=combined_df, x='Model', y='Avg_Node_SMAPE', hue='Experiment', ax=ax)
ax.set_xlabel('Model', fontweight='bold')
ax.set_ylabel('Avg Node SMAPE (%)', fontweight='bold')
ax.set_title('SMAPE Distribution by Model', fontsize=12, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Experiment')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Key Insights Summary

In [ ]:
# Generate key insights
print("=" * 80)
print(" " * 25 + "KEY INSIGHTS SUMMARY")
print("=" * 80)

# 1. Best overall models
print("\n1. BEST PERFORMING MODELS:")
print("-" * 80)
for horizon in sorted(combined_df['Horizon'].unique()):
    h_data = combined_df[combined_df['Horizon'] == horizon]
    best_row = h_data.loc[h_data['Avg_Node_SMAPE'].idxmin()]
    print(f"   Horizon {horizon:2d} week(s): {best_row['Model']:15s} ({best_row['Experiment']}) - "
          f"SMAPE: {best_row['Avg_Node_SMAPE']:.2f}%")

# 2. Experiment comparison
print("\n2. EXPERIMENT COMPARISON:")
print("-" * 80)
exp1_wins = len(comparison_df[comparison_df['Better_Exp'] == 'Exp1'])
exp2_wins = len(comparison_df[comparison_df['Better_Exp'] == 'Exp2'])
print(f"   Exp1 wins: {exp1_wins}/{len(comparison_df)} cases ({exp1_wins/len(comparison_df)*100:.1f}%)")
print(f"   Exp2 wins: {exp2_wins}/{len(comparison_df)} cases ({exp2_wins/len(comparison_df)*100:.1f}%)")
print(f"   Average improvement when Exp1 wins: {comparison_df[comparison_df['Better_Exp']=='Exp1']['SMAPE_Diff'].mean():.2f}%")
print(f"   Average improvement when Exp2 wins: {-comparison_df[comparison_df['Better_Exp']=='Exp2']['SMAPE_Diff'].mean():.2f}%")

# 3. Horizon difficulty
print("\n3. FORECAST DIFFICULTY BY HORIZON:")
print("-" * 80)
for horizon in sorted(combined_df['Horizon'].unique()):
    h_data = combined_df[combined_df['Horizon'] == horizon]
    print(f"   Horizon {horizon:2d} week(s): Avg SMAPE = {h_data['Avg_Node_SMAPE'].mean():.2f}% "
          f"(range: {h_data['Avg_Node_SMAPE'].min():.2f}% - {h_data['Avg_Node_SMAPE'].max():.2f}%)")

# 4. Most consistent models
print("\n4. MOST CONSISTENT MODELS (across horizons):")
print("-" * 80)
for exp in ['Exp1', 'Exp2']:
    exp_consistency = consistency_df[consistency_df['Experiment'] == exp].head(3)
    print(f"   {exp}:")
    for _, row in exp_consistency.iterrows():
        print(f"      {row['Model']:15s} - CV: {row['CV_SMAPE']:.2f}%")

# 5. Worst performers
print("\n5. MODELS NEEDING IMPROVEMENT:")
print("-" * 80)
worst_overall = combined_df.groupby('Model')['Avg_Node_SMAPE'].mean().sort_values(ascending=False).head(3)
for model, smape in worst_overall.items():
    print(f"   {model:15s} - Avg SMAPE across all experiments: {smape:.2f}%")

print("\n" + "=" * 80)

## 8. Advanced Visualizations

In [ ]:
# Scatter plot: SMAPE vs RMSE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, horizon in enumerate(sorted(combined_df['Horizon'].unique())):
    ax = axes[idx]
    
    horizon_data = combined_df[combined_df['Horizon'] == horizon]
    
    # Separate by experiment
    for exp, marker, color in [('Exp1', 'o', 'steelblue'), ('Exp2', 's', 'coral')]:
        exp_data = horizon_data[horizon_data['Experiment'] == exp]
        ax.scatter(exp_data['Avg_Node_SMAPE'], exp_data['Avg_Node_RMSE'], 
                  s=100, alpha=0.7, marker=marker, color=color, label=exp)
        
        # Add model labels
        for _, row in exp_data.iterrows():
            ax.annotate(row['Model'][:4], 
                       (row['Avg_Node_SMAPE'], row['Avg_Node_RMSE']),
                       fontsize=7, alpha=0.7, ha='center')
    
    ax.set_xlabel('Avg Node SMAPE (%)', fontweight='bold')
    ax.set_ylabel('Avg Node RMSE', fontweight='bold')
    ax.set_title(f'Horizon {horizon} week(s): SMAPE vs RMSE', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate correlation
print("\n=== CORRELATION: SMAPE vs RMSE ===")
for horizon in sorted(combined_df['Horizon'].unique()):
    h_data = combined_df[combined_df['Horizon'] == horizon]
    corr = h_data['Avg_Node_SMAPE'].corr(h_data['Avg_Node_RMSE'])
    print(f"Horizon {horizon} week(s): {corr:.4f}")

In [ ]:
# Performance matrix: Model rank across horizons
print("=== MODEL RANK MATRIX (by SMAPE - 1 is best) ===\n")

rank_matrix = []
for exp in ['Exp1', 'Exp2']:
    print(f"\n{exp}:")
    print("-" * 70)
    
    exp_ranks = {}
    for model in sorted(combined_df['Model'].unique()):
        exp_ranks[model] = []
    
    for horizon in sorted(combined_df['Horizon'].unique()):
        h_data = combined_df[(combined_df['Horizon'] == horizon) & (combined_df['Experiment'] == exp)]
        h_data = h_data.sort_values('Avg_Node_SMAPE')
        
        for rank, (_, row) in enumerate(h_data.iterrows(), 1):
            exp_ranks[row['Model']].append(rank)
    
    # Calculate average rank
    model_avg_ranks = []
    for model, ranks in exp_ranks.items():
        if ranks:
            avg_rank = np.mean(ranks)
            model_avg_ranks.append((model, ranks, avg_rank))
    
    model_avg_ranks.sort(key=lambda x: x[2])
    
    print(f"{'Model':<15} {'H=1':>5} {'H=6':>5} {'H=12':>5} {'Avg Rank':>10}")
    print("-" * 70)
    for model, ranks, avg_rank in model_avg_ranks:
        if len(ranks) == 3:
            print(f"{model:<15} {ranks[0]:>5} {ranks[1]:>5} {ranks[2]:>5} {avg_rank:>10.2f}")

## 9. Conclusions and Recommendations

In [ ]:
# Generate final recommendations
print("=" * 80)
print(" " * 20 + "CONCLUSIONS AND RECOMMENDATIONS")
print("=" * 80)

# Find overall best models
overall_best = combined_df.groupby('Model')['Avg_Node_SMAPE'].mean().sort_values().head(3)

print("\n1. RECOMMENDED MODELS FOR DEPLOYMENT:")
print("-" * 80)
for rank, (model, avg_smape) in enumerate(overall_best.items(), 1):
    model_data = combined_df[combined_df['Model'] == model]
    best_horizon = model_data.loc[model_data['Avg_Node_SMAPE'].idxmin()]['Horizon']
    worst_horizon = model_data.loc[model_data['Avg_Node_SMAPE'].idxmax()]['Horizon']
    
    print(f"\n   {rank}. {model}")
    print(f"      Overall Avg SMAPE: {avg_smape:.2f}%")
    print(f"      Best at: Horizon {best_horizon} weeks")
    print(f"      Weakest at: Horizon {worst_horizon} weeks")

print("\n2. SHORT-TERM FORECASTING (1 week):")
print("-" * 80)
h1_best = combined_df[combined_df['Horizon'] == 1].nsmallest(3, 'Avg_Node_SMAPE')
for _, row in h1_best.iterrows():
    print(f"   • {row['Model']} ({row['Experiment']}): {row['Avg_Node_SMAPE']:.2f}% SMAPE")

print("\n3. MEDIUM-TERM FORECASTING (6 weeks):")
print("-" * 80)
h6_best = combined_df[combined_df['Horizon'] == 6].nsmallest(3, 'Avg_Node_SMAPE')
for _, row in h6_best.iterrows():
    print(f"   • {row['Model']} ({row['Experiment']}): {row['Avg_Node_SMAPE']:.2f}% SMAPE")

print("\n4. LONG-TERM FORECASTING (12 weeks):")
print("-" * 80)
h12_best = combined_df[combined_df['Horizon'] == 12].nsmallest(3, 'Avg_Node_SMAPE')
for _, row in h12_best.iterrows():
    print(f"   • {row['Model']} ({row['Experiment']}): {row['Avg_Node_SMAPE']:.2f}% SMAPE")

print("\n5. EXPERIMENT SELECTION:")
print("-" * 80)
if exp1_wins > exp2_wins:
    print(f"   ✓ Exp1 configuration is generally better ({exp1_wins}/{len(comparison_df)} wins)")
else:
    print(f"   ✓ Exp2 configuration is generally better ({exp2_wins}/{len(comparison_df)} wins)")

print("\n6. MODELS TO AVOID:")
print("-" * 80)
worst_models = combined_df.groupby('Model')['Avg_Node_SMAPE'].mean().sort_values(ascending=False).head(2)
for model, avg_smape in worst_models.items():
    print(f"   ✗ {model}: Avg SMAPE {avg_smape:.2f}% across all horizons")

print("\n" + "=" * 80)
print("\n📊 Analysis complete! Use the visualizations above to explore detailed patterns.")
print("=" * 80)